In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from Bio import SeqIO
from tqdm import tqdm
import json
import os
import logging
import numpy as np
import matplotlib.patches as patches
import random
from scipy.stats import mannwhitneyu

# Import all our custom pipeline modules
from instanexus import preprocessing
from instanexus import assembly
from instanexus import visualization
from instanexus import helpers

# Set up logging to see the pipeline's progress
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

In [2]:
pd.options.display.max_colwidth = None

In [3]:
os.chdir('../../../../')

print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/marcor/Desktop/projects/InstaNexus


In [4]:
FIGURES_DIR = Path("figures")
print(FIGURES_DIR)

figures


In [5]:
# Path to the new raw data you want to test
INPUT_CSV = "inputs/nb6.csv"


# Base folder for all results
BASE_OUTPUT_FOLDER = "outputs_notebook"

# Paths to your static database files$
METADATA_PATH = "json/sample_metadata.json"
CONTAMINANTS_PATH = "fasta/contaminants.fasta"

# --- 2. Define Pipeline Parameters ---
RUN_NAME = Path(INPUT_CSV).stem
REFERENCE_MODE = True
CHAIN = ""

# Filtering params
CONFIDENCE_THRESHOLD = 0.8
#MASS_ERR_LIMIT = 20
MIN_LENGTH = 7
#MAX_IRT_ERROR = 60
#MIN_ENTROPY = 1
#PROSIT_FILTER = True
FDR_THRESHOLD = 0.1
#Z_SCORE_THRESHOLD = -0.5

# Assembly params
ASSEMBLY_MODE = "dbg_weighted"
KMER_SIZE = 6
MIN_OVERLAP = 3
SIZE_THRESHOLD = 10

MIN_IDENTITY = 1
MAX_MISMATCHES = 0

# Clustering params
MIN_SEQ_ID = 0.85
COVERAGE = 0.8

In [6]:
base_output_folder = Path(BASE_OUTPUT_FOLDER) / RUN_NAME

# Build the unique experiment folder name
folder_name_parts = [f"{ASSEMBLY_MODE}"]

if CONFIDENCE_THRESHOLD is not None:
    folder_name_parts.append(f"c{CONFIDENCE_THRESHOLD}")

if "dbg" in ASSEMBLY_MODE:
    folder_name_parts.append(f"ks{KMER_SIZE}")

folder_name_parts.append(f"mo{MIN_OVERLAP}")
folder_name_parts.append(f"ts{SIZE_THRESHOLD}")

if REFERENCE_MODE:
    folder_name_parts.extend([f"mi{MIN_IDENTITY}", f"mm{MAX_MISMATCHES}"])

run_folder_name = "_".join(folder_name_parts)
experiment_folder = base_output_folder / run_folder_name


# --- Define a Run ID for logging ---
run_id_str = f"[{RUN_NAME} @ {run_folder_name}]"

logger.info(f"Pipeline starting for run: {run_id_str}")
logger.info(f"All results will be saved to: {experiment_folder}")

2025-12-22 16:35:37,086 [INFO] Pipeline starting for run: [nb6 @ dbg_weighted_c0.8_ks6_mo3_ts10_mi1_mm0]
2025-12-22 16:35:37,087 [INFO] All results will be saved to: outputs_notebook/nb6/dbg_weighted_c0.8_ks6_mo3_ts10_mi1_mm0


In [7]:
sample_metadata = preprocessing.get_sample_metadata(
    run=RUN_NAME, 
    chain=CHAIN, 
    json_path=METADATA_PATH
)

In [8]:
proteases = sample_metadata["proteases"]
protein = sample_metadata["protein"]
protein_norm = preprocessing.normalize_sequence(protein)

In [9]:
print(f"Sample uses proteases: {proteases}")
print(f"Protein sequence length: {len(protein)} amino acids")
print(f"Normalized protein sequence: {protein_norm}")

Sample uses proteases: ['Vesuvius', 'Krakatoa', 'Elastase', 'Trypsin', 'GluC', 'Chymotrypsin', 'Papain', 'ProteinaseK', 'Thermolysin']
Protein sequence length: 156 amino acids
Normalized protein sequence: QVQLQESGGGLVQPGGSLRLSCTASLNLFSLNAMGWYRQAPGKQRELVAALTSGGSTNYADSVKGRFTLSRDNAKSTVYLQMNSLKPEDTAVYYCHAEGPFNLATKEQYDYWGQGTQVTVSSAAADYKDHDGDYKDHDLDYKDDDDKGAAHHHHHH


In [10]:
original_data = pd.read_csv(INPUT_CSV)

In [11]:
original_data.columns

Index(['spectrum_id', 'experiment_name', 'prediction_untokenised',
       'instanovo_token_log_probabilities', 'calibrated_confidence',
       'psm_q_value', 'delta_mass_ppm', 'Mass Error',
       'is_missing_prosit_features', 'ion_match_intensity', 'ion_matches',
       'iRT', 'iRT error', 'is_missing_irt_error', 'predicted iRT', 'margin',
       'entropy', 'z-score'],
      dtype='object')

In [12]:
cols_to_keep = [
    'experiment_name',
    'prediction_untokenised',
    'instanovo_token_log_probabilities',
    'calibrated_confidence',    
    'psm_q_value',
    'delta_mass_ppm',
    'Mass Error',               
    'is_missing_prosit_features', 
    'ion_match_intensity',
    'ion_matches',
    'iRT',
    'iRT error',
    'is_missing_irt_error',
    'predicted iRT',
    'margin',
    'entropy',
    'z-score'
    ]

data = original_data[cols_to_keep].copy()

In [13]:
data.rename(columns={'calibrated_confidence': 'conf'}, inplace=True)

In [14]:
data["protease"] = data["experiment_name"].apply(
    lambda name: preprocessing.extract_protease(name, proteases)
)

protease_col = data.pop("protease")
data.insert(data.columns.get_loc("prediction_untokenised") + 1, "protease", protease_col)

In [15]:
data = data.dropna(subset=["prediction_untokenised"])

In [16]:
data["cleaned_preds"] = data["prediction_untokenised"].apply(preprocessing.remove_modifications)

# move cleaned_preds next to prediction_untokenised
cleaned_preds_col = data.pop("cleaned_preds")
data.insert(data.columns.get_loc("prediction_untokenised") + 1, "cleaned_preds", cleaned_preds_col)

In [17]:
cleaned_psms = data["cleaned_preds"].tolist()

In [18]:
filtered_psms = preprocessing.filter_contaminants(
    cleaned_psms, RUN_NAME , CONTAMINANTS_PATH
)

In [19]:
data = data[data["cleaned_preds"].isin(filtered_psms)]

In [20]:
data.drop(columns=['prediction_untokenised'], inplace=True)

In [21]:
data["mapped"] = data["cleaned_preds"].apply(
    lambda x: "True" if x in protein_norm else "False"
)

In [22]:
data = data[data['cleaned_preds'].str.len() >= MIN_LENGTH]

# exclude psms greather than 20
MAX_LENGHT = 20
data = data[data['cleaned_preds'].str.len() <= MAX_LENGHT]

In [23]:
# show me value counts of mapped vs unmapped
data['mapped'].value_counts()

mapped
False    45180
True      1731
Name: count, dtype: int64

In [24]:
def add_quantification_data(df_main, run_name, fdr_threshold, inputs_folder="inputs"):
    """
    Filters df_main by FDR, then looks for a quantification file ({run_name}_quant_scores.csv).
    Merges the abundance data into the filtered dataframe.
    """
    if fdr_threshold is not None:
        if "psm_q_value" in df_main.columns:
            initial_len = len(df_main)
            df_main = df_main[df_main['psm_q_value'] <= fdr_threshold].copy()
            logger.info(f"FDR Filter applied inside merge function: {initial_len} -> {len(df_main)} rows (<= {fdr_threshold})")
        else:
            logger.warning("FDR threshold provided but 'psm_q_value' column missing. Skipping filter.")

    quant_file_name = f"{run_name}_quant_scores.csv"
    quant_file_path = Path(inputs_folder) / quant_file_name
    
    if not quant_file_path.exists():
        logger.warning(f"Quantification file NOT FOUND: {quant_file_path}")
        logger.warning("Skipping abundance merging. 'peptide_abundance' will be missing.")
        return df_main

    logger.info(f"Found quantification file: {quant_file_path}")
    
    try:
        df_quant = pd.read_csv(quant_file_path)
        
        if "cleaned_preds" not in df_quant.columns or "total_abundance_norm" not in df_quant.columns:
            logger.warning(f"Quantification file format error. Missing columns in {quant_file_path}")
            return df_main

        df_quant_summed = df_quant.groupby('cleaned_preds', as_index=False)['total_abundance_norm'].sum()  
        df_quant_summed.rename(columns={'total_abundance_norm': 'peptide_abundance'}, inplace=True) 
        df_merged = pd.merge(df_main, df_quant_summed, on='cleaned_preds', how='left')
        df_merged['peptide_abundance'] = df_merged['peptide_abundance'].fillna(0)
        
        logger.info(f"Quantification data merged successfully. Output rows: {len(df_merged)}")
        return df_merged

    except Exception as e:
        logger.error(f"Error merging quantification data: {e}")
        return df_main

In [25]:
data_abundance = add_quantification_data(data, RUN_NAME, FDR_THRESHOLD)

2025-12-22 16:35:55,795 [INFO] FDR Filter applied inside merge function: 46911 -> 325 rows (<= 0.1)
2025-12-22 16:35:55,796 [INFO] Found quantification file: inputs/nb6_quant_scores.csv
2025-12-22 16:35:55,899 [INFO] Quantification data merged successfully. Output rows: 325


In [26]:
sequences = data_abundance['cleaned_preds'].tolist()

In [ ]:
import importlib
importlib.reload(assembly)

In [27]:
assembler = assembly.Assembler(
    mode="dbg_weighted",
    kmer_size=7,
    #min_overlap=3,
    size_threshold=10,
    min_weight=2,
    #refine_rounds=5
)

In [28]:
scaffolds = assembler.run(sequences=sequences, df_full=data_abundance)

2025-12-22 16:36:12,025 [INFO] [Assembler] Running DBG weighted (k=7, min_weight=2)
Assembling contigs: 100%|██████████| 144/144 [00:00<00:00, 138877.85it/s]
2025-12-22 16:36:12,128 [INFO] DBG produced 11 initial contigs.


In [29]:
mapped_scaffolds = visualization.process_protein_contigs_scaffold(
    scaffolds, protein_norm, 10, 0.8)

In [ ]:
def mapping_substitutions_seaborn(
    mapped_sequences,
    prot_seq,
    category,
    config_json_path="color_config.json",
    output_folder=".",
    output_file=None,
    show_figure=False,
):
    visualization.set_publication_style()
    
    try:
        with open(config_json_path, 'r') as f:
            color_data = json.load(f)
        main_color = color_data.get(category, {}).get("scaffold", "#1f78b4")
    except Exception as e:
        main_color = "#1f78b4"

    fig_width, fig_height = visualization.get_figsize(width_ratio=3)
    
    common_height = 0.3
    track_spacing = 0.45
    base_y_offset = 0.6

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    ax.add_patch(patches.Rectangle(
        (0, 0), len(prot_seq), common_height,
        linewidth=0, facecolor='#e6f0ef', zorder=0
    ))

    tracks = {}
    colors = {
        "match": main_color,
        "mismatch": "#b30000",
        "D_to_N": "#000000",
        "E_to_Q": "#A8A29E",
    }

    for seq, mapping in tqdm(mapped_sequences, desc=f"Mapping {category}"):
        start_index, end_index, mismatches, _ = mapping
        
        placed = False
        for track_num in sorted(tracks.keys()):
            if not any(max(s, start_index) < min(e, end_index) for s, e in tracks[track_num]):
                tracks[track_num].append((start_index, end_index))
                current_track_num = track_num
                placed = True
                break
        
        if not placed:
            current_track_num = len(tracks)
            tracks[current_track_num] = [(start_index, end_index)]

        current_y = base_y_offset + (current_track_num * track_spacing)
        
        ax.add_patch(patches.Rectangle(
            (start_index, current_y), end_index - start_index, common_height,
            linewidth=0.8, edgecolor='white', facecolor=colors["match"], alpha=0.9
        ))

        for mismatch in mismatches:
            abs_index = start_index + mismatch
            if abs_index >= len(prot_seq) or mismatch >= len(seq): continue

            ref_aa = prot_seq[abs_index]
            query_aa = seq[mismatch]

            if query_aa == "D" and ref_aa == "N":
                mut_color = colors["D_to_N"]
            elif query_aa == "E" and ref_aa == "Q":
                mut_color = colors["E_to_Q"]
            else:
                mut_color = colors["mismatch"]

            ax.add_patch(patches.Rectangle(
                (abs_index, current_y), 1, common_height,
                linewidth=0, facecolor=mut_color, zorder=10 
            ))

    ax.set_xlim(0, len(prot_seq))
    max_y = base_y_offset + (len(tracks) * track_spacing) + 0.2
    ax.set_ylim(-0.1, max_y)
    
    ax.set_xlabel("Residue position")
    ax.set_yticks([])
    sns.despine(left=True)

    legend_labels = [
        patches.Patch(color=colors["match"], label=f"Match"),
        patches.Patch(color=colors["mismatch"], label="Mismatch"),
        patches.Patch(color=colors["D_to_N"], label="D \u2192 N"),
        patches.Patch(color=colors["E_to_Q"], label="E \u2192 Q"),
    ]
    ax.legend(handles=legend_labels, loc='upper center', 
              bbox_to_anchor=(0.5, 1.25), ncol=4)

    plt.tight_layout()

    if output_file:
        try:
            os.makedirs(output_folder, exist_ok=True)
            save_path = os.path.join(output_folder, output_file)
            plt.savefig(save_path, bbox_inches='tight')
        except PermissionError:
            plt.savefig(output_file, bbox_inches='tight')

    if show_figure:
        plt.show()
    plt.close()

In [30]:
mapped_scaffolds

[('SLKPEDTAVY', (83, 93, [], 1.0)),
 ('RQAPGKQREL', (37, 47, [], 1.0)),
 ('VAALTSGGSTNYADSVKGR', (47, 66, [], 1.0)),
 ('SLNAMGWYRQAPGK', (29, 43, [], 1.0)),
 ('SVKGRFTLSRDNAKSTVYLQMNSLK', (61, 86, [], 1.0)),
 ('HAEGPFNLATKEQYDYWGQGTQVTVSS', (95, 122, [], 1.0)),
 ('DLDYKDDDDKGAAHHHHHH', (137, 156, [], 1.0)),
 ('ESGGGLVQPGGSLR', (5, 19, [], 1.0)),
 ('QMNSLKVEDTAVY', (80, 93, [6], 0.9230769230769231)),
 ('RQAPGKERELVA', (37, 49, [6], 0.9166666666666666)),
 ('DYKDHDGDYKDH', (125, 137, [], 1.0))]

In [31]:
FIGURES_DIR

PosixPath('figures')

In [33]:
visualization.mapping_sequences(
    mapped_sequences=mapped_scaffolds,
    prot_seq=protein_norm,
    category="nanobodies",
    config_json_path="colors.json",
    output_folder=FIGURES_DIR,
    output_file=f"supp_fig5b_{RUN_NAME}_substitutions_mapping.svg",
    show_figure=True
)

Mapping nanobodies: 100%|██████████| 11/11 [00:00<00:00, 3201.54it/s]
/Users/marcor/Desktop/projects/InstaNexus/src/instanexus/visualization.py:1322: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Boxplot coverage

In [ ]:
def plot_coverage_boxplot_seaborn_layered(
    file_path, 
    output_image=f'{FIGURES_DIR}/fig4a_coverage_boxplot_layered.svg',
    ratio=1
):
    visualization.set_publication_style()

    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        data = {
            'assembly_method': ['greedy (Contigs)']*15 + ['greedy (Scaffolds)']*15,
            'coverage': np.concatenate([
                np.random.normal(0.85, 0.08, 15), 
                np.random.normal(0.92, 0.05, 15)
            ])
        }
        df = pd.DataFrame(data)

    df['Type'] = df['assembly_method'].apply(
        lambda x: 'Contigs' if 'Contigs' in x else 'Scaffolds'
    )
    
    colors = {"Contigs": "#a6cee3", "Scaffolds": "#1f78b4"}

    fig_w, fig_h = visualization.get_figsize(width_ratio=ratio)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    sns.boxplot(
        data=df, 
        x='Type', 
        y='coverage', 
        palette=colors,
        width=0.4,
        showfliers=False, 
        linewidth=1.2,
        ax=ax,
        boxprops=dict(alpha=0.8),
        whiskerprops=dict(color='#333333'),
        capprops=dict(color='#333333'),
        medianprops=dict(color='#333333', linewidth=1.5)
    )
    
    sns.stripplot(
        data=df, 
        x='Type', 
        y='coverage', 
        palette=colors,     
        size=5,
        jitter=0.2,
        linewidth=0.8,
        alpha=0.6,          
        ax=ax,
        edgecolor="#333333"               
    )

    ax.set_ylabel('Protein coverage', fontweight='normal')
    ax.set_xlabel('', fontweight='normal')
    
    if df['coverage'].max() <= 1.0:
        ax.set_ylim(0.4, 1.05)
    else:
        ax.set_ylim(0, 105)

    sns.despine(ax=ax, offset=0, trim=False)

    if output_image:
        Path(os.path.dirname(output_image)).mkdir(parents=True, exist_ok=True)
        plt.savefig(output_image, format='svg', bbox_inches='tight')
    
    plt.show()

In [ ]:
plot_coverage_boxplot_seaborn_layered('outputs/_summary_tables/dbg_weighted/best_results_Nanobodies_dbg_weighted.csv')

## Depth PMSs

In [ ]:
def plot_psm_depth_standardized(
    reference_seq: str, 
    peptides: list, 
    cdrs: dict, 
    output_file: str = 'fig_4C_matplotlib.svg',
    ratio=2
):
    visualization.set_publication_style()
        
    def chars_equal(a, b):
        return (a in ['L', 'I'] and b in ['L', 'I']) or a == b

    def find_all_occurrences(ref, pep):
        starts = []
        for pos in range(len(ref) - len(pep) + 1):
            if all(chars_equal(ref[pos+i], pep[i]) for i in range(len(pep))):
                starts.append(pos)
        return starts

    depth = np.zeros(len(reference_seq), dtype=int)
    for pep in peptides:
        found_indices = find_all_occurrences(reference_seq, pep)
        for start_idx in found_indices:
            end_idx = start_idx + len(pep)
            depth[start_idx:end_idx] += 1
            
    fig_w, fig_h = visualization.get_figsize(width_ratio=ratio)
    
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    
    x = np.arange(len(reference_seq))
    
    BLUE_COLOR = '#4A90E2'
    ax.plot(x, depth, color=BLUE_COLOR, linewidth=1.2, label='PSM Depth')
    ax.fill_between(x, depth, color=BLUE_COLOR, alpha=0.15)
    
    highlight_colors = {
        "CDR1": "#FFB347",
        "CDR2": "#77DD77",
        "CDR3": "#89CFF0"
    }
    
    max_y = depth.max() * 1.1 if depth.max() > 0 else 1.0
    
    for label, (start, end) in cdrs.items():
        ax.axvspan(start-1, end, color=highlight_colors.get(label, 'gray'), alpha=0.2, zorder=0)
        
        ax.text((start-1 + end)/2, max_y * 0.95, label, 
                ha='center', va='top', fontweight='normal', color='black')

    ax.set_title('PSM depth across protein sequence', fontweight='normal', pad=15)
    ax.set_xlabel('Amino acid position', fontweight='normal')
    ax.set_ylabel('Depth (PSMs)', fontweight='normal')
    
    sns.despine(ax=ax, offset=0, trim=False)
    
    ax.set_xlim(0, len(reference_seq))
    ax.set_ylim(0, max_y)
    
    if output_file:
        Path(os.path.dirname(output_file)).mkdir(parents=True, exist_ok=True)
        plt.savefig(output_file, format='svg', bbox_inches='tight')
    
    plt.show()

In [ ]:
plot_psm_depth_standardized(
    reference_seq=protein_norm, 
    peptides=data_abundance['cleaned_preds'].tolist(), 
    cdrs={
        "CDR1": (26, 34),
        "CDR2": (50, 66),
        "CDR3": (99, 112)
    }, 
    output_file=f'{FIGURES_DIR}/fig4b_{RUN_NAME}_psm_depth.svg'
)

## Consensuns sequences aligned

In [ ]:
def plot_consensus_alignment_standardized(
    reference_seq: str,
    scaffolds: list,
    cdrs: dict,
    colors: dict,
    output_file: str = "../../../../figures/fig4e_consensus_alignment.svg",
    show_figure: bool = True
):

    def chars_equal(a, b):
        return (a in ['L', 'I'] and b in ['L', 'I']) or a == b

    def find_alignment_offset(ref, seq):
        for pos in range(len(ref) - len(seq) + 1):
            if all(chars_equal(ref[pos+i], seq[i]) for i in range(len(seq))):
                return pos
        return ref.find(seq)

    char_w = 0.12 
    calculated_width = (len(reference_seq) * char_w) + 2.0
    calculated_height = 1.5 + (len(scaffolds) * 0.5)

    fig, ax = plt.subplots(figsize=(calculated_width, calculated_height), layout='constrained')
    
    ax.axis('off')
    
    FONT_SIZE = 10 
    FONT_FAMILY = 'Arial'
    Y_REF = 0
    Y_STEP = -1.0
    
    
    ax.text(-2, Y_REF, "Reference", ha='right', va='center', fontsize=FONT_SIZE+1, fontweight='bold', color='#2c3e50')
    
    for i, char in enumerate(reference_seq):
        ax.text(i, Y_REF, char, ha='center', va='center', fontsize=FONT_SIZE, fontfamily=FONT_FAMILY)

    for cdr_name, (start, end) in cdrs.items():
        x0 = (start - 1) - 0.5
        width = (end - start + 1)
        
        rect_ref = patches.Rectangle(
            (x0, Y_REF - 0.3), width, 0.6, 
            facecolor=colors.get(cdr_name, 'gray'), alpha=0.3, zorder=0
        )
        ax.add_patch(rect_ref)
        
        ax.text(
            (start - 1 + end)/2 - 0.5, Y_REF + 0.5, cdr_name, 
            ha='center', va='bottom', fontsize=8, fontweight='bold', color=colors.get(cdr_name, 'black')
        )

    for idx, (scaf_name, scaf_seq) in enumerate(scaffolds):
        y_pos = Y_REF + ((idx + 1) * Y_STEP)
        
        offset = find_alignment_offset(reference_seq, scaf_seq)
        if offset == -1: continue
            
        ax.text(-2, y_pos, scaf_name, ha='right', va='center', fontsize=FONT_SIZE+1, color='#2c3e50')
        
        for i, char in enumerate(scaf_seq):
            x_pos = offset + i
            ax.text(x_pos, y_pos, char, ha='center', va='center', fontsize=FONT_SIZE, fontfamily=FONT_FAMILY)
            
        for cdr_name, (start, end) in cdrs.items():
            cdr_start_0, cdr_end_0 = start - 1, end - 1
            scaf_start, scaf_end = offset, offset + len(scaf_seq) - 1
            ov_start = max(cdr_start_0, scaf_start)
            ov_end = min(cdr_end_0, scaf_end)
            
            if ov_start <= ov_end:
                rect_scaf = patches.Rectangle(
                    (ov_start - 0.5, y_pos - 0.3), ov_end - ov_start + 1, 0.6, 
                    facecolor=colors.get(cdr_name, 'gray'), alpha=0.3, zorder=0
                )
                ax.add_patch(rect_scaf)

    y_ruler = Y_REF + ((len(scaffolds) + 0.5) * Y_STEP)
    for i in range(0, len(reference_seq), 10):
        label_num = i + 1
        ax.text(i, y_ruler, str(label_num), ha='center', va='top', fontsize=7, color='gray')

    ax.set_xlim(-15, len(reference_seq) + 2)
    ax.set_ylim(y_ruler - 1, Y_REF + 1.5)
    
    os.makedirs(os.path.dirname(output_file) if os.path.dirname(output_file) else '.', exist_ok=True)
    
    plt.savefig(output_file, dpi=300, bbox_inches=None)
    
    if show_figure:
        plt.show()

In [ ]:
plot_consensus_alignment_standardized()

## Barplot composite score

In [ ]:
def plot_barplot_composite_coverage_scores(
    csv_path, 
    category, 
    config_json_path="json/colors.json", 
    output_file_coverage=None, 
    output_file_composite=None
):

    visualization.set_publication_style()
    df = pd.read_csv(csv_path)
    
    try:
        with open(config_json_path, 'r') as f:
            color_data = json.load(f)
        palette = [
            color_data.get(category.lower(), {}).get("contig", "#a6cee3"),
            color_data.get(category.lower(), {}).get("scaffold", "#1f78b4")
        ]
    except Exception:
        palette = ["#a6cee3", "#1f78b4"]

    df['Type'] = df['assembly_method'].apply(lambda x: "Contigs" if "Contigs" in x else "Scaffolds")
    
    try:
        df['sample_num'] = df['sample'].str.extract(r'(\d+)').astype(int)
        df = df.sort_values('sample_num')
    except Exception:
        pass 

    plots_to_generate = [
        ("coverage", "Coverage", output_file_coverage),
        ("composite_score", "Composite Score", output_file_composite)
    ]

    for col_name, y_label, out_file in plots_to_generate:
        fig_width, fig_height = visualization.get_figsize(width_ratio=3)
        plt.figure(figsize=(fig_width, fig_height))

        ax = sns.barplot(
            data=df,
            x='sample',
            y=col_name,
            hue='Type',
            palette=palette,
            edgecolor='black',
            linewidth=0.8
        )

        ax.set_ylim(0, 1.05)
        ax.set_xlabel("Samples")
        ax.set_ylabel(y_label)
        
        plt.legend(title="", loc='upper right', frameon=False)
        
        sns.despine()
        plt.tight_layout()

        if out_file:
            os.makedirs(os.path.dirname(out_file), exist_ok=True)
            plt.savefig(out_file, bbox_inches='tight')
            print(f"Saved {y_label} plot to {out_file}")
        
        plt.show()
        plt.close()


In [ ]:
plot_barplot_composite_coverage_scores(csv_path="outputs/_summary_tables/dbg_weighted/best_results_Nanobodies_dbg_weighted.csv", category="Nanobodies", output_file="figures/fig4c_composite_scores_nb.svg")

In [ ]:
plot_barplot_composite_coverage_scores(
    csv_path="outputs/_summary_tables/dbg_weighted/best_results_Nanobodies_dbg_weighted.csv",
    category="Nanobodies",
    output_file_coverage="figures/supp_fig5a_nanobodies_coverage.svg",
    output_file_composite="figures/fig4c_nanobodies_composite.svg"
)